# Hops=1 vs hops=8 calibration probe (CPU, 0 GPU quota)

Runs `tools/probe_hops_calibration.py` on the real GGUF models to answer two
questions before switching `_fill()` to a tighter replay cap:

1. **Hit-rate invariance** -- does replaying at `max_tool_hops=1` fire on the
   *same* EXFILTRATION candidates as `max_tool_hops=8`? (Hardware-independent;
   the go/no-go signal.)
2. **Calibration coefficient** -- `coef = elapsed(hops=8) / elapsed(hops=1)`
   per model, so a hops=1 fill measurement can be scaled back to the true
   forced-hops=8 replay cost `REPLAY_SAFE_SIZING` must budget for.

CPU kernel: no GPU offload, so this does not touch the weekly GPU quota.
The bootstrap cell below recreates the repo files the probe needs; the SDK
and model servers come from the attached competition dataset.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys


def find_repo_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/AI-Agent-Security')]
    for candidate in candidates:
        if (candidate / 'tools' / 'probe_hops_calibration.py').exists():
            return candidate
    raise FileNotFoundError('tools/probe_hops_calibration.py not found; run the bootstrap cell first')


ROOT = find_repo_root()
os.chdir(ROOT)
print('repo root:', ROOT)
# CPU kernel: nvidia-smi is absent, and subprocess.run raises FileNotFoundError
# when the executable itself is missing (capture_output cannot catch that), so
# probe for it first.
if shutil.which('nvidia-smi'):
    gpu = subprocess.run(['nvidia-smi', '-L'], text=True, capture_output=True)
    print('gpu:', gpu.stdout.strip() or gpu.stderr.strip())
else:
    print('gpu: no GPU visible (CPU kernel)')
print('cpu count:', os.cpu_count())


In [ ]:
os.environ.setdefault('GPT_OSS_GGUF_REPO', 'unsloth/gpt-oss-20b-GGUF')
os.environ.setdefault('GPT_OSS_GGUF_FILE', 'gpt-oss-20b-Q4_K_M.gguf')
os.environ.setdefault('GEMMA_GGUF_REPO', 'unsloth/gemma-4-26B-A4B-it-GGUF')
os.environ.setdefault('GEMMA_GGUF_FILE', 'gemma-4-26B-A4B-it-UD-Q4_K_M.gguf')

# CPU kernel: install the CPU wheel of llama-cpp-python. With no CUDA backend
# compiled in, the model server's n_gpu_layers=-1 is a no-op and inference runs
# fully on CPU -- which is why this probe costs 0 GPU quota.
os.environ.setdefault('LLAMA_CPP_EXTRA_INDEX_URL', 'https://abetlen.github.io/llama-cpp-python/whl/cpu')

# To run fully offline against attached-dataset weights instead of HF download,
# set these to the .gguf paths before running:
# os.environ['GPT_OSS_MODEL_PATH'] = '/kaggle/input/<dataset>/gpt-oss-20b-Q4_K_M.gguf'
# os.environ['GEMMA_MODEL_PATH'] = '/kaggle/input/<dataset>/gemma-4-26B-A4B-it-UD-Q4_K_M.gguf'
for name in ('GPT_OSS_MODEL_PATH', 'GEMMA_MODEL_PATH'):
    if os.getenv(name):
        print(name, os.getenv(name))


In [ ]:
import importlib.util


def ensure_llama_cpp() -> None:
    if importlib.util.find_spec('llama_cpp') is not None:
        print('llama_cpp already installed')
        return
    extra_index = os.getenv('LLAMA_CPP_EXTRA_INDEX_URL',
                            'https://abetlen.github.io/llama-cpp-python/whl/cpu')
    wheel_cmd = [sys.executable, '-m', 'pip', 'install', '-q', '--prefer-binary',
                 'llama-cpp-python', '--extra-index-url', extra_index]
    print('installing llama-cpp-python (CPU) from', extra_index)
    try:
        subprocess.run(wheel_cmd, check=True)
    except subprocess.CalledProcessError:
        print('prebuilt CPU wheel failed; building llama-cpp-python for CPU from source')
        env = os.environ.copy()
        env['CMAKE_ARGS'] = '-DGGML_CUDA=off'
        env['FORCE_CMAKE'] = '1'
        subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
                        '--force-reinstall', 'llama-cpp-python'], check=True, env=env)
    if importlib.util.find_spec('llama_cpp') is None:
        raise ModuleNotFoundError('llama_cpp')


ensure_llama_cpp()


In [ ]:
import json

cmd = [sys.executable, 'tools/probe_hops_calibration.py',
       '--n', '12', '--models', 'gpt_oss,gemma',
       '--budget-per-model', '3000',
       '--hops', '1,8', '--baseline-hop', '8', '--probe-hop', '1',
       '--out', 'research/results/hops-calibration.latest.json',
       '--raw-out', 'research/results/hops-calibration.raw.jsonl']
print('running:', ' '.join(cmd))
# Exit code 2 means "a candidate that fired at hops=8 dropped out at hops=1" --
# an informative NEGATIVE result, not a crash, so do NOT use check=True here.
proc = subprocess.run(cmd, text=True)
print('probe exit code:', proc.returncode,
      '(0 = hit-rate invariant across all models, 2 = a candidate dropped out at hops=1)')


In [ ]:
import shutil

summary_path = Path('research/results/hops-calibration.latest.json')
summary = json.loads(summary_path.read_text())
print(json.dumps(summary, indent=2, sort_keys=True))

print('\n=== VERDICT ===')
print('hit_rate_invariant_all_models:', summary.get('hit_rate_invariant_all_models'))
for model, a in summary.get('analyses', {}).items():
    inv = a['hit_rate_invariance']
    cal = a['calibration']
    print(f"[{model}] invariant={inv['invariant']} "
          f"fired base/probe={inv['fired_baseline']}/{inv['fired_probe']} "
          f"coef_p50={cal['coef_p50']} coef_mean={cal['coef_mean']} "
          f"(warm base {cal['baseline_warm_mean']}s -> probe {cal['probe_warm_mean']}s)")

# Copy outputs to /kaggle/working root so they are easy to retrieve via kernels_output.
out_dir = Path('/kaggle/working')
if out_dir.exists():
    for p in [summary_path, Path('research/results/hops-calibration.raw.jsonl')]:
        if p.exists() and p.resolve() != (out_dir / p.name).resolve():
            shutil.copy(p, out_dir / p.name)
    print('copied outputs to', out_dir)
